# W2D5 — Pipelines, Regularisation & Cross-Validation — Guided

**Week 2 · Day 5 · Data Engineering & Preprocessing** · Lab

Every score you have reported this week came from **one** train/test split. Today you find out what
that number actually was: one sample from a distribution, quoted as if it were a measurement.

You have compared 0.7915 against 0.7885 and drawn a conclusion from it. By the end of the warm-up you
will know how wide the spread of that kind of number is, and therefore whether a gap of 0.003 was
ever a result at all.

Then three things, in order:

- **`Pipeline`** — the structure that puts preprocessing *inside* the model, so cross-validation
  refits it per fold and leakage becomes hard rather than merely discouraged.
- **Regularisation** — what L2 and L1 do to a model's coefficients, measured rather than described,
  including the case where the answer is "nothing, on this data".
- **Leakage, twice** — once where the bug moves the score by 0.00003 and once where it moves it to a
  perfect 1.0000. One of those is the version that survives code review, and it is not the one you
  expect.

You'll leave with `cv_results.parquet` and `pipeline.joblib` — the first appearance of the artefact
your capstone requires, and where W8D1 picks up.

**Time budget:** ~115 minutes. Sections 1–2 are the lab; Section 3 is a stretch you may finish at home.

<div dir="rtl" align="right">

# الأسبوع ٢ اليوم ٥ — خطوط المعالجة والتنظيم والتحقّق المتقاطع

**الأسبوع ٢ · اليوم ٥ · هندسة البيانات والمعالجة المسبقة** · معمل عملي

كل نتيجة عرضتها هذا الأسبوع جاءت من تقسيم **واحد** إلى تدريب واختبار. واليوم تعرف ما كان ذلك الرقم
فعلًا: عيّنة واحدة من توزيع، مذكورة كأنها قياس.

فقد قارنت ٠٫٧٩١٥ بـ ٠٫٧٨٨٥ وخرجت باستنتاج. وبنهاية التهيئة ستعرف كم يتّسع مدى تفاوت هذا النوع من
الأرقام، ومن ثمّ هل كان فرق ٠٫٠٠٣ نتيجةً أصلًا.

ثم ثلاثة أمور بالترتيب:

- **خطّ المعالجة `Pipeline`** وهو البنية التي تضع المعالجة المسبقة **داخل** النموذج، فيعيد التحقّق
  المتقاطع تدريبها في كل طيّة، ويصبح التسريب صعبًا لا مجرّد أمر غير مستحسن.
- **التنظيم** وما يفعله L2 وL1 بمعاملات النموذج، مقيسًا لا موصوفًا، ومنه الحالة التي يكون الجواب فيها
  «لا شيء، على هذه البيانات».
- **التسريب مرتين**: مرة يُحرّك فيها العيب النتيجة بمقدار ٠٫٠٠٠٠٣، ومرة يرفعها إلى ١٫٠٠٠٠ الكاملة.
  وإحداهما هي النسخة التي تنجو من مراجعة الشيفرة، وهي ليست التي تتوقّعها.

ستخرج بملفَّي `cv_results.parquet` و`pipeline.joblib` — وهو أول ظهور للأثر الذي يطلبه مشروع تخرّجك،
ومنه يبدأ الأسبوع الثامن اليوم الأول.

**الزمن المتوقّع:** نحو ١١٥ دقيقة. القسمان الأول والثاني هما المعمل، والقسم الثالث إضافي يمكن إكماله في المنزل.

</div>

> **This is the guided version.** Most of the code is already here. Fill in the lines marked
> `# TODO`. If you want the full challenge, use the `_blank` version instead.

<div dir="rtl" align="right">

> **هذه النسخة الموجَّهة.** معظم الشيفرة موجودة، وعليك إكمال الأسطر المعلَّمة بـ `# TODO`.
> وإذا أردت التحدّي الكامل فاستخدم نسخة `_blank`.

</div>

## Learning objectives

By the end of this lab you can:

- Report a model's performance as a mean **and** a spread, and say why the mean alone is a claim you
  cannot support.
- Build a `ColumnTransformer` and wrap it with an estimator in a single `Pipeline`.
- Cross-validate with `StratifiedKFold`, and say what plain `KFold` risks and when.
- Read a train-versus-validation curve across regularisation strengths, and name where a model starts
  memorising — or say that it never does.
- Count the coefficients L1 drives to exactly zero, and explain why L2 cannot.
- Recognise that leakage is not detectable from the score, and that this is why `Pipeline` exists.
- Save a fitted pipeline and prove the reloaded copy predicts identically.

<div dir="rtl" align="right">

## أهداف التعلّم

بنهاية هذا المعمل ستكون قادرًا على:

- عرض أداء النموذج بمتوسط **ومدى تفاوت**، وبيان لماذا يكون المتوسط وحده ادعاءً لا تستطيع دعمه.
- بناء `ColumnTransformer` ولفّه مع مُقدِّر في خطّ معالجة `Pipeline` واحد.
- التحقّق المتقاطع بـ `StratifiedKFold`، وبيان ما يخاطر به `KFold` العادي ومتى.
- قراءة منحنى التدريب مقابل التحقّق على شدّات التنظيم، وتحديد أين يبدأ النموذج بالحفظ — أو قول إنه لا يبدأ.
- عدّ المعاملات التي يجعلها L1 أصفارًا تامة، وشرح لماذا لا يستطيع L2 ذلك.
- إدراك أن التسريب غير قابل للكشف من النتيجة، وأن هذا هو سبب وجود `Pipeline`.
- حفظ خطّ معالجة مُدرَّب وإثبات أن النسخة المُعاد تحميلها تتنبّأ تنبّؤًا مطابقًا.

</div>

## About the data

**Dataset:** `telco_churn`, via **W2D1's `features.parquet`** — 7,043 rows × 25 columns

Back to the classification target, deliberately. `revenue` was a regression problem where every fold
looks like every other fold. `Churn` is **26.5% positive**, and that imbalance is what makes the
difference between `KFold` and `StratifiedKFold` something you can measure rather than something you
are told.

If you did not finish W2D1, `load_artefact` falls back to the reference copy in
`shared/solutions_cache/` and prints a note saying so. You are not blocked.

The metric today is **ROC AUC**, not accuracy. W1D4 established why: 73.5% of these customers stayed,
so accuracy starts at 0.735 for a model that does nothing, and a change of one percentage point in
accuracy can mean anything. AUC asks whether the model ranks a churner above a stayer, which is the
question a retention team actually acts on.

**Watch out:** one of the 25 columns is `customerID`, and it has 7,043 distinct values in 7,043 rows.
It is not a feature and it never was. In Section 2 you will use it to build the single most
destructive leak in this course, on purpose, and it will take four lines.

<div dir="rtl" align="right">

## عن البيانات

**مجموعة البيانات:** `telco_churn` عبر ملف اليوم الأول `features.parquet` — ٧٬٠٤٣ صفًا × ٢٥ عمودًا

عودة إلى هدف التصنيف عن قصد. فقد كان `revenue` مسألة انحدار تتشابه فيها كل الطيّات، أما `Churn`
فنسبة الموجب فيه **٢٦٫٥٪**، وهذا الاختلال هو ما يجعل الفرق بين `KFold` و`StratifiedKFold` شيئًا تقيسه
لا شيئًا يُقال لك.

وإن لم تُكمل اليوم الأول فإن `load_artefact` ترجع إلى النسخة المرجعية في `shared/solutions_cache/`
وتطبع ملاحظة بذلك، فأنت غير متعطّل.

والمقياس اليوم هو **المساحة تحت منحنى ROC** لا الدقة. وقد بيّن الأسبوع الأول اليوم الرابع السبب: فقد
بقي ٧٣٫٥٪ من هؤلاء العملاء، فتبدأ الدقة من ٠٫٧٣٥ لنموذج لا يفعل شيئًا، وقد يعني تغيّر نقطة مئوية واحدة
فيها أي شيء. أما المساحة تحت المنحنى فتسأل هل يرتّب النموذج المغادر فوق الباقي، وهذا هو السؤال الذي
يتصرّف فريق الاحتفاظ بناءً عليه.

**انتبه:** أحد الأعمدة الخمسة والعشرين هو `customerID`، وفيه ٧٬٠٤٣ قيمة مختلفة في ٧٬٠٤٣ صفًا. وهو ليس
خاصية ولم يكن كذلك قط. وستستخدمه في القسم الثاني لبناء أشدّ تسريب في هذه الدورة تدميرًا، عن قصد، وفي
أربعة أسطر.

</div>

## Setup

Run the cell below first. It loads W2D1's artefact and fixes the column lists and the cross-validator
that the whole lab shares, so that no comparison in this notebook differs by anything other than the
thing being compared.

<div dir="rtl" align="right">

## الإعداد

شغّل الخلية التالية أولًا. تُحمّل مخرجات اليوم الأول وتُثبّت قوائم الأعمدة والمُقسِّم المتقاطع الذي
يشترك فيه المعمل كله، فلا يختلف أي مقارنة في هذا الدفتر بشيء غير الشيء المُقارَن.

</div>

In [ ]:
# === AIEP portable setup — works locally (Miniconda + uv) and on Google Colab ===============
try:
    import aiep
except ImportError:
    import subprocess, sys
    from pathlib import Path
    # A clone that never ran `uv pip install -e shared/` still has the package on disk —
    # use it before reaching for the network. Colab (no clone) falls through to pip.
    _local = next((p / "shared" for p in [Path.cwd(), *Path.cwd().parents]
                   if (p / "shared" / "aiep").is_dir()), None)
    if _local:
        sys.path.insert(0, str(_local))
    else:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                               "git+https://github.com/0xRush/AIEP_Olo_student.git#subdirectory=shared"])
    import aiep

from aiep.env import ensure, seed_everything, versions
from aiep.data import load_artefact
from aiep.paths import ARTEFACT_DIR
from aiep.checks import check, report

ensure("scikit-learn", "matplotlib", "pyarrow")
seed_everything(42)

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import (KFold, StratifiedKFold, cross_val_score,
                                     cross_validate, train_test_split)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from aiep.viz import use_course_style, PALETTE
use_course_style()

df = pd.read_parquet(load_artefact("features.parquet"))

TARGET = "churn_flag"
NUMERIC = ["tenure", "MonthlyCharges", "TotalCharges", "contract_months", "SeniorCitizen"]
CATEGORICAL = ["Contract", "InternetService", "PaymentMethod", "tenure_bucket",
               "OnlineSecurity", "TechSupport", "PaperlessBilling",
               "gender", "Partner", "Dependents"]

X = df[NUMERIC + CATEGORICAL]
y = df[TARGET]

# One cross-validator, shared by every comparison in this notebook. Same folds every
# time, so two numbers differ only by what they are measuring.
CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
SCORING = "roc_auc"

print(f"{len(df):,} rows | {len(NUMERIC)} numeric + {len(CATEGORICAL)} categorical features")
print(f"churn rate: {y.mean():.1%}")
print("\n", versions())

## Section 1 — Warm-up: the number you have been quoting  (≈25 min)

Everything in this section already works.

The cell below fits the same model on **five different train/test splits** — same data, same
features, same hyperparameters, only the `random_state` of the split changes. Nothing about the model
differs between the five runs.

Run it, then compute the mean and the spread yourself before you read the next cell.

<div dir="rtl" align="right">

## القسم الأول — التهيئة: الرقم الذي كنت تذكره (نحو ٢٥ دقيقة)

كل ما في هذا القسم يعمل أصلًا.

تُدرّب الخليّة أدناه النموذج نفسه على **خمسة تقسيمات مختلفة** إلى تدريب واختبار — البيانات نفسها
والخصائص نفسها والمعاملات نفسها، ولا يتغيّر إلا `random_state` الخاص بالتقسيم. فلا شيء في النموذج
يختلف بين التشغيلات الخمس.

شغّلها ثم احسب المتوسط ومدى التفاوت بنفسك قبل قراءة الخليّة التالية.

</div>

In [ ]:
def make_pipeline(C=1.0, l1=False, extra_numeric=()):
    """A ColumnTransformer plus a LogisticRegression, in one estimator.

    l1=False gives ridge (L2); l1=True gives lasso (L1), which needs liblinear.
    `extra_numeric` lets later tasks add a column without rebuilding this by hand.
    """
    preprocess = ColumnTransformer([
        ("num", Pipeline([("impute", SimpleImputer(strategy="median")),
                          ("scale", StandardScaler())]), NUMERIC + list(extra_numeric)),
        ("cat", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL),
    ])
    return Pipeline([
        ("preprocess", preprocess),
        ("model", LogisticRegression(C=C, l1_ratio=1 if l1 else 0,
                                     solver="liblinear" if l1 else "lbfgs",
                                     max_iter=2000)),
    ])


from sklearn.metrics import roc_auc_score

single_split_scores = []
for state in range(5):
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, stratify=y, random_state=state)
    fitted = make_pipeline().fit(X_train, y_train)
    auc = roc_auc_score(y_test, fitted.predict_proba(X_test)[:, 1])
    single_split_scores.append(auc)
    print(f"random_state={state}:  AUC = {auc:.4f}")

single_split_scores = np.array(single_split_scores)

### Task 1.1 — how wide is that?

Compute the mean, the standard deviation, and the gap between the best and worst of those five.

Then answer the question that the rest of the week depended on: **if you had run this once and
reported the number, which of the five would you have reported?**

<div dir="rtl" align="right">

### المهمة ١٫١ — كم يتّسع ذلك؟

احسب المتوسط والانحراف المعياري والفرق بين أفضل الخمس وأسوئها.

ثم أجب عن السؤال الذي اعتمد عليه بقيّة الأسبوع: **لو شغّلت هذا مرة واحدة وذكرت الرقم، فأي الخمسة كنت
ستذكر؟**

</div>

In [ ]:
print(f"mean        {single_split_scores.mean():.4f}")
print(f"std         {single_split_scores.std():.4f}")
print(f"best        {single_split_scores.max():.4f}   (random_state="
      f"{single_split_scores.argmax()})")
print(f"worst       {single_split_scores.min():.4f}   (random_state="
      f"{single_split_scores.argmin()})")
print(f"best-worst  {single_split_scores.max() - single_split_scores.min():.4f}")

print("\nNothing about the model changed between those five runs.")
print("Every difference you see is the split, and nothing else.")
print("\nNow look back at yesterday: the indicator strategy beat the median strategy by 0.0030.")
print("Compare that to the spread above, and decide what yesterday's comparison established.")

The five scores span **0.027 of AUC**, and yesterday's "result" was a gap of 0.003 — well
inside the range you would get by changing nothing at all except which rows landed in the test set.

Be careful about how far that goes, because there is an over-correction available here and it is also
wrong. Yesterday's comparison is not *worthless*: the three imputation strategies were compared on
the **same split**, so the split's luck was held constant across them, and the ranking is more stable
than three independent numbers would be. What yesterday could not tell you is whether the ranking
survives a different split. That is a question about the spread, and answering it needs more than one
split.

Which is the whole idea of cross-validation, and the rest of the lab.

<div dir="rtl" align="right">

تمتدّ النتائج الخمس على **٠٫٠٢٧ من المساحة تحت المنحنى**، و«نتيجة» الأمس كانت فرقًا مقداره ٠٫٠٠٣،
أي داخل المدى الذي تحصل عليه بلا تغيير أي شيء إلا أي الصفوف وقع في مجموعة الاختبار.

وانتبه إلى حدود هذا الاستنتاج، فهناك مبالغة في التصحيح متاحة وهي خاطئة أيضًا. فمقارنة الأمس ليست **بلا
قيمة**: إذ قُورنت طرائق التعويض الثلاث على **التقسيم نفسه**، فبقي حظّ التقسيم ثابتًا بينها، والترتيب
أثبت من ثلاثة أرقام مستقلّة. وما لم يستطع الأمس إخبارك به هو هل يصمد الترتيب على تقسيم مختلف. وذلك
سؤال عن مدى التفاوت، وجوابه يحتاج أكثر من تقسيم واحد.

وهذه هي فكرة التحقّق المتقاطع كلها، وهي بقيّة المعمل.

</div>

## Section 2 — Core: six tasks  (≈60 min)

1. Build the `Pipeline` — preprocessing and estimator as one object.
2. Cross-validate it. Report the mean **and** the standard deviation.
3. Swap `StratifiedKFold` for plain `KFold` and look at what each fold contains.
4. Sweep the regularisation strength. Plot train against validation. Then count L1's zeros.
5. **Break it on purpose, twice.** One bug moves the score by 0.00003. The other moves it to 1.0000.
6. Save the fitted pipeline, reload it, and prove the copy predicts identically.

<div dir="rtl" align="right">

## القسم الثاني — الأساسي: ستّ مهام (نحو ٦٠ دقيقة)

١. ابنِ خطّ المعالجة: المعالجة المسبقة والمُقدِّر كائنًا واحدًا.
٢. تحقّق منه متقاطعًا، واذكر المتوسط **والانحراف المعياري**.
٣. استبدل `StratifiedKFold` بـ `KFold` العادي وانظر ما في كل طيّة.
٤. امسح شدّة التنظيم، وارسم التدريب مقابل التحقّق، ثم عُدّ أصفار L1.
٥. **اكسِره عن قصد مرتين.** عيب يُحرّك النتيجة ٠٫٠٠٠٠٣، وآخر يرفعها إلى ١٫٠٠٠٠.
٦. احفظ خطّ المعالجة المُدرَّب، وأعد تحميله، وأثبت أن النسخة تتنبّأ تنبّؤًا مطابقًا.

</div>

### Task 2.1 — one object, not four

Until now your preprocessing has been a sequence of cells above a `.fit()` call. That works, and it
has one fatal property: **nothing enforces the order.** The imputer, the scaler and the encoder are
fitted whenever you happened to run their cells, on whatever rows were in scope at the time.

A `Pipeline` makes preprocessing part of the model. Then `.fit()` fits every step in order on
whatever rows it was given, and `.predict()` applies them in the same order to new rows. And because
cross-validation calls `.fit()` once per fold, the preprocessing is refitted **per fold**, using only
that fold's training rows — which is the thing you would otherwise have to remember to do by hand,
five times, correctly.

Two steps here: a `ColumnTransformer` that treats numeric and categorical columns differently, then
the estimator.

<div dir="rtl" align="right">

### المهمة ٢٫١ — كائن واحد لا أربعة

كانت معالجتك المسبقة حتى الآن سلسلة خلايا فوق استدعاء `.fit()`. وهذا يعمل، وله خاصية قاتلة واحدة:
**لا شيء يفرض الترتيب.** فالمُعوِّض والمُقيِّس والمُرمِّز تُدرَّب متى صادف أنك شغّلت خلاياها، وعلى أي
صفوف كانت في النطاق حينها.

ويجعل `Pipeline` المعالجة المسبقة جزءًا من النموذج. فيدرّب `.fit()` كل خطوة بالترتيب على الصفوف التي
أُعطيها، ويطبّق `.predict()` الخطوات بالترتيب نفسه على صفوف جديدة. ولأن التحقّق المتقاطع ينادي `.fit()`
مرة لكل طيّة، تُعاد تدريب المعالجة المسبقة **في كل طيّة** بصفوف تدريب تلك الطيّة وحدها — وهو ما كان
عليك أن تتذكّر فعله يدويًا خمس مرات وبصواب.

وهنا خطوتان: `ColumnTransformer` يعامل الأعمدة الرقمية والفئوية معاملة مختلفة، ثم المُقدِّر.

</div>

In [ ]:

# TODO: Build the ColumnTransformer: impute + scale the numeric columns, one-hot the rest.
# مهمة: ابنِ `ColumnTransformer`: تعويض وتقييس للأعمدة الرقمية، وترميز أحادي للبقية.

# TODO: Wrap it with a LogisticRegression in a single two-step Pipeline.
# مهمة: لُفَّه مع انحدار لوجستي في خطّ معالجة واحد من خطوتين.

print("steps:", list(pipeline.named_steps))
print(f"\nfitting on all {len(X):,} rows just to see it work …")
pipeline.fit(X, y)
print(f"features after preprocessing: "
      f"{pipeline.named_steps['preprocess'].transform(X).shape[1]}")

### Task 2.2 — cross-validate it, and report the spread

`cross_val_score` splits the data into five folds, and for each one fits the pipeline on four and
scores it on the fifth. Five scores, and every one of them is out-of-sample.

Report the **mean and the standard deviation**. Reporting the mean alone is the single most common
way a model's performance is overstated in practice — `0.84 ± 0.01` and `0.84 ± 0.15` are completely
different claims, and only one of them describes a model you would deploy.

<div dir="rtl" align="right">

### المهمة ٢٫٢ — تحقّق متقاطعًا واذكر مدى التفاوت

تقسم `cross_val_score` البيانات خمس طيّات، وتدرّب خطّ المعالجة في كل مرة على أربع وتقيّمه على الخامسة.
فخمس نتائج، وكل واحدة منها خارج العيّنة.

واذكر **المتوسط والانحراف المعياري**. فذكر المتوسط وحده هو أشيع طريقة يُبالَغ بها في أداء النموذج
عمليًا: إذ إن `0.84 ± 0.01` و`0.84 ± 0.15` ادعاءان مختلفان تمامًا، وواحد منهما فقط يصف نموذجًا تنشره.

</div>

In [ ]:

# TODO: Cross-validate the pipeline with the shared StratifiedKFold and AUC.
# مهمة: تحقّق متقاطعًا من خطّ المعالجة بـ `StratifiedKFold` المشترك والمساحة تحت المنحنى.
cv_scores = ...

print("fold scores:", np.round(cv_scores, 4).tolist())
print(f"\nmean {cv_scores.mean():.4f}   std {cv_scores.std():.4f}")
print(f"report it as: AUC = {cv_scores.mean():.3f} +/- {cv_scores.std():.3f}")
print(f"\nthe warm-up's five single splits spread "
      f"{single_split_scores.max() - single_split_scores.min():.4f}; "
      f"these five spread {cv_scores.max() - cv_scores.min():.4f}")

Note what cross-validation did and did not buy you.

It did **not** shrink the spread. Five folds vary about as much as five random splits did, because
the variation was never a flaw in the method — it is a real property of estimating performance from
1,400 test rows. What it bought you is that the spread is now **visible and reportable**, and that
every row is used for testing exactly once instead of 20% of the data doing all the work.

So the honest report is `0.843 ± 0.011`. Anything you compare against it has to clear that band
before you call it an improvement.

<div dir="rtl" align="right">

ولاحظ ما اشتراه لك التحقّق المتقاطع وما لم يشتره.

فهو **لم** يُقلّص مدى التفاوت. إذ تتفاوت الطيّات الخمس بمقدار قريب من تفاوت خمسة تقسيمات عشوائية، لأن
التفاوت لم يكن عيبًا في الطريقة قط، بل هو خاصية حقيقية لتقدير الأداء من ألف وأربعمئة صف اختبار. وإنما
اشترى لك أن مدى التفاوت صار **مرئيًا وقابلًا للعرض**، وأن كل صف يُستخدَم للاختبار مرة واحدة بالضبط بدل
أن يقوم خُمس البيانات بالعمل كله.

فالعرض الصادق هو `0.843 ± 0.011`. وأي شيء تقارنه به يجب أن يتجاوز هذا النطاق قبل أن تسمّيه تحسّنًا.

</div>

### Task 2.3 — what plain `KFold` risks

Swap `StratifiedKFold` for plain `KFold` and look at the churn rate **inside each fold**.

You will find the folds are fine. That is not because `KFold` is safe — it is because this file
happens to arrive in an order unrelated to the target. So the second half of this task is to make the
file arrive the way files usually do: **sorted**. Sort by the target and look again.

Data arrives sorted all the time. Exported by status, appended by date, grouped by branch, ordered by
whatever the source system's primary key happens to be. `KFold` without shuffling preserves that
order into the folds.

<div dir="rtl" align="right">

### المهمة ٢٫٣ — ما يخاطر به `KFold` العادي

استبدل `StratifiedKFold` بـ `KFold` العادي وانظر إلى نسبة المغادرة **داخل كل طيّة**.

وستجد الطيّات سليمة. وليس ذلك لأن `KFold` آمن، بل لأن هذا الملف يصل مصادفةً بترتيب لا علاقة له بالهدف.
فالنصف الثاني من هذه المهمة هو أن تجعل الملف يصل كما تصل الملفات عادةً: **مرتّبًا**. رتّب بحسب الهدف
وانظر من جديد.

فالبيانات تصل مرتّبة دائمًا: مُصدَّرة بحسب الحالة، أو مُلحَقة بحسب التاريخ، أو مجموعة بحسب الفرع، أو
مرتّبة بحسب أي مفتاح رئيسي في النظام المصدر. و`KFold` بلا خلط يُبقي ذلك الترتيب في الطيّات.

</div>

In [ ]:

# TODO: Print the churn rate of each plain-KFold test fold, in the file's own order.
# مهمة: اطبع نسبة المغادرة في كل طيّة اختبار من `KFold` العادي، بترتيب الملف نفسه.

# TODO: Sort the frame by the target, then run plain KFold again.
# مهمة: رتّب الجدول بحسب الهدف ثم شغّل `KFold` العادي من جديد.

# TODO: Confirm StratifiedKFold on the same sorted frame gives balanced folds.
# مهمة: تأكّد أن `StratifiedKFold` على الجدول المرتّب نفسه يعطي طيّات متوازنة.

On the file as it arrives, plain `KFold` gives churn rates between 0.258 and 0.271. Perfectly usable
— **by luck.**

Sort the file first and three of the five folds contain no churners at all, one contains nothing but
churners, and one is a mixture. On a fold with one class, AUC is undefined: there is no pair of a
positive and a negative to rank. Accuracy on those folds is a flat 100% or 0%, and a mean of five
numbers where three of them are meaningless is not a performance estimate, it is arithmetic performed
on nonsense.

Two things follow:

1. **Stratify whenever the target is a class.** It costs one word and removes an entire failure mode.
   `StratifiedKFold` on the sorted frame gives 0.265 in every fold, because it splits within each
   class.
2. **The bug you are guarding against is not `KFold`.** It is the assumption that your row order
   carries no information. It usually does, and you almost never checked.

<div dir="rtl" align="right">

على الملف كما يصل، يعطي `KFold` العادي نسب مغادرة بين ٠٫٢٥٨ و٠٫٢٧١، وهي صالحة تمامًا — **بالحظّ**.

ورتّب الملف أولًا فتجد ثلاثًا من الطيّات الخمس لا تحتوي مغادرًا واحدًا، وواحدة لا تحتوي إلا مغادرين،
وواحدة مختلطة. وعلى طيّة فيها فئة واحدة تكون المساحة تحت المنحنى غير معرَّفة: فلا يوجد زوج من موجب
وسالب لترتيبهما. والدقة على تلك الطيّات إمّا ١٠٠٪ أو ٠٪، ومتوسط خمسة أرقام ثلاثة منها بلا معنى ليس
تقديرًا للأداء بل حسابًا على لا شيء.

ويتبع ذلك أمران:

١. **طبّق التقسيم الطبقي كلما كان الهدف فئة.** فهو يكلّف كلمة واحدة ويُزيل نمط فشل كاملًا. ويعطي
   `StratifiedKFold` على الجدول المرتّب نسبة ٠٫٢٦٥ في كل طيّة، لأنه يقسم داخل كل فئة.
٢. **والعيب الذي تحتاط منه ليس `KFold`**، بل هو افتراض أن ترتيب صفوفك لا يحمل معلومة. وهو يحملها
   عادةً، وأنت لم تتحقّق تقريبًا أبدًا.

</div>

### Task 2.4 — the regularisation sweep

Regularisation adds a penalty on the size of the coefficients, so the model is no longer minimising
error alone — it is minimising error *plus* a price for complexity. `C` is the inverse strength:
**small `C` means a heavy penalty**, large `C` means almost none.

Sweep six values of `C` with L2, cross-validating each, and record the **train and validation** score
for every fold. Plot both against `C` on a log axis. The gap between the two curves is overfitting,
measured.

Then run L1 and count how many coefficients came back **exactly** zero.

**Do the sweep twice**: once on all 7,043 rows, and once on a 200-row subsample. The second one is
not padding — the first is going to show you almost nothing, and understanding why is the point.

<div dir="rtl" align="right">

### المهمة ٢٫٤ — مسح التنظيم

يضيف التنظيم عقوبة على حجم المعاملات، فلا يعود النموذج يُصغّر الخطأ وحده بل يُصغّر الخطأ **مع** ثمن
للتعقيد. و`C` هو مقلوب الشدّة: **فصغر `C` يعني عقوبة ثقيلة**، وكبره يعني ألّا عقوبة تقريبًا.

امسح ستّ قيم لـ `C` بتنظيم L2، مع التحقّق المتقاطع لكل واحدة، وسجّل نتيجة **التدريب والتحقّق** لكل
طيّة. وارسم الاثنين مقابل `C` على محور لوغاريتمي. والفرق بين المنحنيين هو فرط المطابقة مقيسًا.

ثم شغّل L1 وعُدّ كم معاملًا عاد صفرًا **تامًّا**.

**وامسح مرتين**: مرة على الصفوف السبعة آلاف وثلاثة والأربعين كلها، ومرة على عيّنة من مئتي صف. والثانية
ليست حشوًا: فالأولى لن تُظهر لك شيئًا تقريبًا، وفهم السبب هو المقصود.

</div>

In [ ]:

C_VALUES = [0.001, 0.01, 0.1, 1, 10, 100]

# A deliberately small subsample. 200 rows is where a 33-feature model can memorise.
small_idx = np.random.default_rng(7).choice(len(df), size=200, replace=False)
X_small, y_small = X.iloc[small_idx], y.iloc[small_idx]

# TODO: Sweep C on both sample sizes, collecting one row per fold.
# مهمة: امسح قيم C على حجمي العيّنة، وجمّع صفًا لكل طيّة.

# TODO: Average over folds and print both sweeps side by side.
# مهمة: خُذ المتوسط على الطيّات واطبع المسحين جنبًا إلى جنب.

In [ ]:

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# TODO: One panel per sample size: train and validation against C, log x, best C marked.
# مهمة: لوحة لكل حجم عيّنة: التدريب والتحقّق مقابل C بمحور لوغاريتمي، مع تعليم أفضل C.

plt.show()

Two sweeps, two completely different stories, and the pair is the lesson.

**On all 7,043 rows, nothing happens.** Train runs 0.842 to 0.846, validation 0.841 to 0.843, and the
gap never exceeds 0.0033. There is no value of `C` that meaningfully overfits and none that
meaningfully underfits, because a linear model with 33 coefficients cannot memorise 5,600 training
rows however hard you let it try. If you had only run this sweep, the honest conclusion would be
**"regularisation strength does not matter on this problem"** — and reporting that is better work
than inventing a story about the fourth decimal.

**On 200 rows, the textbook curve appears.** Train climbs from 0.813 to 0.863 as the penalty comes
off, while validation *peaks at C = 0.01 and then falls* to 0.761. The gap opens from 0.026 to 0.102.
That is bias–variance with numbers attached: at heavy penalty the model is too simple and both scores
are low together; at no penalty it fits noise in the training folds that does not exist in the
validation fold.

**The thing that changed was not the model. It was the amount of data.** Regularisation matters when
your model has enough capacity relative to your sample to memorise. On 7,000 rows this model does
not; on 200 it does. Which tells you where to look first when someone asks whether they should tune
it — count the rows before you build the grid.

<div dir="rtl" align="right">

مسحان وقصّتان مختلفتان تمامًا، والزوج نفسه هو الدرس.

**على الصفوف السبعة آلاف وثلاثة والأربعين كلها لا يحدث شيء.** فالتدريب يمتدّ من ٠٫٨٤٢ إلى ٠٫٨٤٦
والتحقّق من ٠٫٨٤١ إلى ٠٫٨٤٣، ولا يتجاوز الفرق ٠٫٠٠٣٣. فلا قيمة لـ `C` تُفرط المطابقة إفراطًا ذا معنى
ولا قيمة تُقصّر تقصيرًا ذا معنى، لأن نموذجًا خطّيًا بثلاثة وثلاثين معاملًا لا يستطيع حفظ خمسة آلاف
وستّمئة صف تدريب مهما تركته يحاول. ولو لم تُشغّل إلا هذا المسح لكان الاستنتاج الصادق أن **«شدّة
التنظيم لا تهمّ في هذه المسألة»** — وعرض ذلك عمل أفضل من اختلاق قصة عن المنزلة العشرية الرابعة.

**وعلى مئتي صف يظهر المنحنى المدرسي.** فالتدريب يصعد من ٠٫٨١٣ إلى ٠٫٨٦٣ مع رفع العقوبة، أما التحقّق
فيبلغ **قمّته عند `C` يساوي ٠٫٠١ ثم يهبط** إلى ٠٫٧٦١. ويتّسع الفرق من ٠٫٠٢٦ إلى ٠٫١٠٢. وهذا هو
الانحياز مقابل التشتّت بأرقام: فمع العقوبة الثقيلة يكون النموذج أبسط من اللازم فتنخفض النتيجتان معًا،
ومع انعدام العقوبة يطابق ضوضاءً في طيّات التدريب لا وجود لها في طيّة التحقّق.

**والذي تغيّر ليس النموذج بل كمية البيانات.** فالتنظيم يهمّ حين تكون سعة نموذجك كافية بالنسبة إلى
عيّنتك للحفظ. وعلى سبعة آلاف صف لا يستطيع هذا النموذج، وعلى مئتين يستطيع. وهذا يخبرك أين تنظر أولًا
حين يسألك أحد هل يضبط التنظيم: عُدّ الصفوف قبل أن تبني الشبكة.

</div>

Now L1. Both penalties shrink coefficients; only L1 sets them to **exactly** zero, which makes it a
feature selector as well as a regulariser. L2 shrinks towards zero and arrives asymptotically — it
will give you 0.0001, never 0.

Count the zeros at each strength, and note what the validation score does while the model is being
emptied out.

<div dir="rtl" align="right">

والآن L1. فكلا التنظيمين يُقلّص المعاملات، وL1 وحده يجعلها أصفارًا **تامة**، فيكون مُنتقيًا للخصائص
ومُنظِّمًا في الوقت نفسه. أما L2 فيُقلّص نحو الصفر ويصل إليه اقترابًا فحسب: فيعطيك ٠٫٠٠٠١ ولا يعطيك صفرًا.

عُدّ الأصفار عند كل شدّة، ولاحظ ما تفعله نتيجة التحقّق أثناء إفراغ النموذج.

</div>

In [ ]:

# TODO: For each C: fit L1 on all rows, count exact zeros, and cross-validate it.
# مهمة: لكل C: درّب L1 على كل الصفوف، وعُدّ الأصفار التامة، وتحقّق منه متقاطعًا.

print(pd.DataFrame(l1_report).to_string(index=False, float_format=lambda v: f"{v:.4f}"))
print(f"\nat the strongest penalty (C={C_VALUES[0]}), L1 zeroed "
      f"{zeros_at_strongest} of {l1_report[0]['of']} coefficients")

At `C = 0.001` L1 zeroes **32 of 33** coefficients and the AUC drops to 0.500 — the model has one
feature left and cannot rank anything. That is the extreme end, and it is worth seeing: regularisation
taken far enough does not degrade a model gracefully, it deletes it.

The useful part is the middle. At `C = 0.1`, L1 zeroes 14 of 33 and still scores 0.842 — within noise
of the full 33-coefficient model's 0.843. **Fourteen of your features were carrying nothing**, and L1
found that out without you writing a selection loop.

That is the practical use of L1: not usually as your final model, but as a question. Fit it, read
which coefficients survived, and go and look at the ones that did not.

<div dir="rtl" align="right">

عند `C` يساوي ٠٫٠٠١ يُصفّر L1 **اثنين وثلاثين من ثلاثة وثلاثين** معاملًا وتهبط المساحة تحت المنحنى
إلى ٠٫٥٠٠ — فلم تبقَ للنموذج إلا خاصية واحدة ولا يستطيع ترتيب شيء. وهذا هو الطرف الأقصى ويستحق
المشاهدة: فالتنظيم إذا بُولغ فيه لا يُضعف النموذج تدريجيًا بل يحذفه.

والمفيد هو الوسط. فعند `C` يساوي ٠٫١ يُصفّر L1 أربعة عشر من ثلاثة وثلاثين ويحقّق ٠٫٨٤٢، أي داخل ضوضاء
نتيجة النموذج الكامل ٠٫٨٤٣. فـ**أربعة عشر من خصائصك لم تكن تحمل شيئًا**، ووجد L1 ذلك دون أن تكتب حلقة
انتقاء.

وهذا هو الاستخدام العملي لـ L1: لا كنموذجك النهائي عادةً بل كسؤال. درّبه، واقرأ أي المعاملات نجا، ثم
اذهب وانظر في التي لم تنجُ.

</div>

### Task 2.5 — break it on purpose, twice

Now the point of the whole day. Two versions of the same bug — **preprocessing fitted before the
split instead of inside each fold** — and they behave completely differently.

**Bug A: fit the scaler on everything.** Transform the full dataset with a `StandardScaler` fitted on
all 7,043 rows, then cross-validate the bare estimator on the result. Every training fold has now
been scaled using statistics that included its own validation fold.

**Bug B: target-encode `customerID` on everything.** Replace each customer with the mean churn rate
of that customer, computed across all rows — then cross-validate.

Predict both deltas before you run it, and write your predictions down. One of them will surprise
you, and it matters which.

<div dir="rtl" align="right">

### المهمة ٢٫٥ — اكسِره عن قصد مرتين

والآن جوهر اليوم كله. نسختان من العيب نفسه — **معالجة مسبقة مُدرَّبة قبل التقسيم لا داخل كل طيّة** —
وسلوكهما مختلف تمامًا.

**العيب الأول: درّب المُقيِّس على كل شيء.** حوّل البيانات كلها بـ `StandardScaler` مُدرَّب على الصفوف
السبعة آلاف وثلاثة والأربعين، ثم تحقّق متقاطعًا من المُقدِّر المجرّد على الناتج. فكل طيّة تدريب صارت
مُقيَّسة بمقاييس شملت طيّة تحقّقها نفسها.

**والعيب الثاني: رمّز `customerID` بالهدف على كل شيء.** استبدل كل عميل بمتوسط نسبة مغادرته محسوبًا على
كل الصفوف، ثم تحقّق متقاطعًا.

توقّع الفرقين قبل التشغيل واكتب توقّعك. فأحدهما سيفاجئك، والمهم أيّهما.

</div>

In [ ]:

honest = cross_val_score(make_pipeline(), X, y, cv=CV, scoring=SCORING)

# TODO: Bug A — fit the preprocessing on the whole dataset, then cross-validate.
# مهمة: العيب الأول — درّب المعالجة المسبقة على البيانات كلها ثم تحقّق متقاطعًا.

# TODO: Bug B — target-encode customerID using every row's label, then cross-validate.
# مهمة: العيب الثاني — رمّز `customerID` بالهدف مستخدمًا تسمية كل صف، ثم تحقّق متقاطعًا.

print(f"honest pipeline            AUC = {honest.mean():.6f} +/- {honest.std():.4f}")
print(f"bug A: scaler on all rows  AUC = {bug_a.mean():.6f} +/- {bug_a.std():.4f}   "
      f"delta {bug_a.mean() - honest.mean():+.6f}")
print(f"bug B: ID target-encoded   AUC = {bug_b.mean():.6f} +/- {bug_b.std():.4f}   "
      f"delta {bug_b.mean() - honest.mean():+.6f}")

print(f"\nrows per customerID: {len(df) / df['customerID'].nunique():.2f}")

Read those two deltas, because between them they contain the reason `Pipeline` exists.

**Bug A moved the score by +0.00003.** Fitting the scaler on 7,043 rows instead of 5,634 changes each
column's mean and standard deviation in the fifth decimal place, so the transformation is
substantively the same one and the model cannot exploit it. Run it with a different `random_state`
and the delta sometimes comes out **negative**. This bug is, on this dataset, undetectable.

That is not a licence to write it. It is the more disturbing finding of the two: **you cannot detect
leakage by checking whether the score went up.** Bug A is genuinely wrong — the validation folds
helped build the transformation — and it happens to be harmless here only because n is large and a
mean is a stable statistic. Change the preprocessing step to something less stable, or shrink the
data, and the same line of code costs you a model. You will not get a warning either way.

**Bug B moved the score to 1.0000, with a standard deviation of 0.0000.** There is exactly one row
per `customerID`, so "the mean churn rate of this customer" *is* this customer's label. The feature
is `y`, renamed. Every fold scores perfectly because the answer was written into the input.

And notice what makes Bug B visible: not the leakage, but the **magnitude**. A model that returns a
flawless 1.0000 on held-out data is not credible on its face, which is the same alarm W2D3 rang at
`R² = 1.0000`. If Bug B had involved a column with 50 rows per customer instead of 1, the score would
have landed at a plausible 0.91 and you would have shipped it.

So the defence cannot be vigilance about scores. It has to be structural, and that is the whole
argument for `Pipeline`: put every fitted step inside the estimator, and cross-validation refits them
per fold whether you remembered the problem or not.

<div dir="rtl" align="right">

اقرأ هذين الفرقين، فبينهما سبب وجود `Pipeline`.

**حرّك العيب الأول النتيجة بمقدار +٠٫٠٠٠٠٣.** فتدريب المُقيِّس على سبعة آلاف صف بدل خمسة آلاف وستّمئة
يغيّر متوسط كل عمود وانحرافه في المنزلة العشرية الخامسة، فيكون التحويل هو نفسه جوهريًا ولا يستطيع
النموذج استغلاله. وشغّله ببذرة عشوائية أخرى فيخرج الفرق **سالبًا** أحيانًا. فهذا العيب على هذه البيانات
غير قابل للكشف.

وليس ذلك إذنًا بكتابته، بل هو أكثر النتيجتين إثارةً للقلق: **لا تستطيع كشف التسريب بالنظر هل ارتفعت
النتيجة.** فالعيب الأول خاطئ فعلًا — إذ شاركت طيّات التحقّق في بناء التحويل — وإنما صار غير ضارّ هنا
لأن حجم العيّنة كبير ولأن المتوسط مقياس مستقرّ. فغيّر خطوة المعالجة إلى شيء أقل استقرارًا، أو صغّر
البيانات، فيكلّفك السطر نفسه نموذجًا. ولن يأتيك تحذير في الحالتين.

**ورفع العيب الثاني النتيجة إلى ١٫٠٠٠٠ بانحراف معياري ٠٫٠٠٠٠.** إذ يوجد صف واحد بالضبط لكل
`customerID`، فيكون «متوسط نسبة مغادرة هذا العميل» **هو** تسمية هذا العميل. فالخاصية هي `y` باسم آخر.
وتحقّق كل طيّة نتيجة كاملة لأن الإجابة كُتبت في المُدخل.

ولاحظ ما جعل العيب الثاني مرئيًا: ليس التسريب بل **المقدار**. فالنموذج الذي يُعيد ١٫٠٠٠٠ بلا نقص على
بيانات محجوزة غير معقول من ظاهره، وهذا هو الجرس نفسه الذي قرعه اليوم الثالث عند `R²` يساوي واحدًا. ولو
كان العيب الثاني على عمود فيه خمسون صفًا لكل عميل بدل صف واحد لهبطت النتيجة إلى ٠٫٩١ المعقولة ولكنت
سلّمتها.

فالدفاع لا يمكن أن يكون التيقّظ للنتائج، بل يجب أن يكون بنيويًا، وهذه هي حجّة `Pipeline` كلها: ضع كل
خطوة مُدرَّبة داخل المُقدِّر، فيُعيد التحقّق المتقاطع تدريبها في كل طيّة سواء تذكّرت المشكلة أم لا.

</div>

### Task 2.6 — save it, reload it, and prove it

A fitted `Pipeline` is one object containing the imputer's medians, the scaler's means, the encoder's
category lists and the model's coefficients. `joblib` writes all of it to a single file.

Save it, load it into a **fresh variable**, and check that the reloaded copy produces the same
predictions on five held-out rows. Not similar predictions — **identical**, exactly, because nothing
about a saved model should be approximate.

This is the artefact your capstone has to produce, and W8D1 starts by loading one.

<div dir="rtl" align="right">

### المهمة ٢٫٦ — احفظه وأعد تحميله وأثبته

خطّ المعالجة المُدرَّب كائن واحد يحتوي وسائط المُعوِّض ومتوسطات المُقيِّس وقوائم فئات المُرمِّز ومعاملات
النموذج. و`joblib` يكتب كل ذلك في ملف واحد.

احفظه، وحمّله في **متغيّر جديد**، وتحقّق أن النسخة المُعاد تحميلها تُنتج التنبّؤات نفسها على خمسة صفوف
محجوزة. لا تنبّؤات متشابهة بل **مطابقة تمامًا**، لأنه لا ينبغي أن يكون شيء في نموذج محفوظ تقريبيًا.

وهذا هو الأثر الذي على مشروع تخرّجك إنتاجه، ويبدأ الأسبوع الثامن اليوم الأول بتحميل واحد منه.

</div>

In [ ]:

X_fit, X_holdout, y_fit, y_holdout = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)

# TODO: Fit the pipeline on the fitting half and predict probabilities for five held-out rows.
# مهمة: درّب خطّ المعالجة على نصف التدريب وتنبّأ باحتمالات خمسة صفوف محجوزة.

# TODO: Save it with joblib, then load it back into a different variable.
# مهمة: احفظه بـ `joblib` ثم أعد تحميله في متغيّر مختلف.

print(f"saved {model_path} ({model_path.stat().st_size / 1024:.1f} KB)")
print(f"\nbefore saving: {np.round(before_saving, 6).tolist()}")
print(f"after loading:  {np.round(after_loading, 6).tolist()}")
print(f"\nbit-for-bit identical: {np.array_equal(before_saving, after_loading)}")
print(f"steps in the reloaded object: {list(reloaded_pipeline.named_steps)}")

## Section 3 — Stretch: write the email  (≈30 min)

Open-ended. Lower expectation of completeness — get through the first part, and treat the second as
homework if you run out of time.

A colleague sends you this, and they are pleased:

> *"Good news on the churn model — I got the AUC up to 1.00 on cross-validation. Turns out
> customer-level history was the missing signal. Five-fold, stratified, standard deviation 0.0000.
> I think we're ready to ship it to the retention team."*

They ran your Bug B. Every technical detail in that message is true.

**Write the reply.** That paragraph is the deliverable, and it is harder than it looks, because
almost everything about the message is defensible: they did cross-validate, they did stratify, they
did report a spread, and their number is real. What they got wrong is upstream of all of it.

A reply worth sending has four things in it:

1. **The specific diagnosis.** Not "I think there might be leakage" — name the column, name why, and
   name the row count that makes it fatal (one row per `customerID`).
2. **The evidence they can run themselves.** What is the one command that shows it? A reply the
   recipient has to take on faith just moves the disagreement.
3. **What the honest number is** — 0.843 ± 0.011 — and that this is not a failure. It is a genuinely
   useful model that ranks a churner above a stayer 84% of the time.
4. **No blame.** They made the most common mistake in applied machine learning and they made it while
   doing three things right. A reply that reads as a correction gets argued with; one that reads as
   "here is what I would check" gets fixed.

Then the second part, which has no clean answer:

> Bug A moved the score by 0.00003 and is undetectable. Your colleague would never have caught it,
> and neither would you. **So what do you actually do about the class of bugs you cannot measure?**

Write your answer. "Use a `Pipeline`" is the beginning of it, not the end — pipelines do not protect
you from a leaking *feature*, only from a leaking *fit*. What else is in your defence?

**Link to your capstone:** the report rubric asks you to justify your evaluation protocol, and
`pipeline.joblib` is a required artefact. Both of them are this section.

<div dir="rtl" align="right">

## القسم الثالث — الإضافي: اكتب الرسالة (نحو ٣٠ دقيقة)

قسم مفتوح، ولا يُتوقّع إكماله بالكامل — أنجز الجزء الأول واعتبر الثاني واجبًا منزليًا إن ضاق الوقت.

يرسل لك زميل هذا وهو مسرور:

> *«خبر جيّد عن نموذج المغادرة: رفعت المساحة تحت المنحنى إلى ١٫٠٠ في التحقّق المتقاطع. تبيّن أن تاريخ
> العميل كان الإشارة الناقصة. خمس طيّات، طبقيّة، والانحراف المعياري ٠٫٠٠٠٠. أظنّنا جاهزين لتسليمه
> لفريق الاحتفاظ.»*

لقد شغّل عيبك الثاني. وكل تفصيل تقني في تلك الرسالة صحيح.

**اكتب الردّ.** فتلك الفقرة هي المُنتَج المطلوب، وهي أصعب مما تبدو، لأن كل شيء في الرسالة تقريبًا قابل
للدفاع: فقد تحقّق متقاطعًا فعلًا، وطبّق التقسيم الطبقي فعلًا، وذكر مدى التفاوت فعلًا، ورقمه حقيقي. وما
أخطأ فيه سابق لكل ذلك.

والردّ الذي يستحق الإرسال فيه أربعة أشياء:

١. **تشخيص محدّد.** لا «أظنّ أن هناك تسريبًا» بل سمِّ العمود، وسمِّ السبب، وسمِّ عدد الصفوف الذي يجعله
   قاتلًا (صف واحد لكل `customerID`).
٢. **دليل يستطيع تشغيله بنفسه.** فما هو الأمر الواحد الذي يُظهر ذلك؟ فالردّ الذي على متلقّيه أن يأخذه
   على الثقة لا ينقل الخلاف إلا نقلًا.
٣. **الرقم الصادق** وهو ٠٫٨٤٣ ± ٠٫٠١١، وأن هذا ليس فشلًا: فهو نموذج مفيد فعلًا يرتّب المغادر فوق
   الباقي في ٨٤٪ من الحالات.
٤. **بلا لوم.** فقد ارتكب أشيع خطأ في تعلّم الآلة التطبيقي، وارتكبه وهو يُحسن ثلاثة أمور. والردّ الذي
   يُقرأ تصحيحًا يُجادَل فيه، والذي يُقرأ «هذا ما كنت سأتحقّق منه» يُصلَح.

ثم الجزء الثاني الذي لا إجابة نظيفة له:

> حرّك العيب الأول النتيجة ٠٫٠٠٠٠٣ وهو غير قابل للكشف. ولم يكن زميلك سيكتشفه ولا كنت أنت.
> **فما تفعله فعلًا بصنف العيوب التي لا تستطيع قياسها؟**

اكتب إجابتك. فقولك «استخدم `Pipeline`» بداية الجواب لا نهايته: إذ لا تحميك خطوط المعالجة من **خاصية**
مُسرِّبة بل من **تدريب** مُسرِّب. فما بقيّة دفاعك؟

**الصلة بمشروعك:** تطلب كرّاسة تقييم التقرير أن تبرّر بروتوكول تقييمك، وملف `pipeline.joblib` أثر
مطلوب. وكلاهما هذا القسم.

</div>

In [ ]:

# TODO: Build the evidence: rows per ID, and the leaking feature's correlation with y.
# مهمة: ابنِ الدليل: عدد الصفوف لكل معرّف، وارتباط الخاصية المُسرِّبة بـ y.

# TODO: Now do the encoding honestly — inside each fold — and show where the score lands.
# مهمة: والآن نفّذ الترميز تنفيذًا صادقًا داخل كل طيّة، وأظهر أين تستقرّ النتيجة.

**Your reply:** _(the four things: the diagnosis, the evidence they can run, the honest number, and
no blame.)_

> …

**And your answer to the second question:** _(what do you do about bugs that do not move the score?)_

<div dir="rtl" align="right">

**ردّك:** _(الأمور الأربعة: التشخيص، والدليل الذي يستطيع تشغيله، والرقم الصادق، وبلا لوم.)_

> …

**وإجابتك عن السؤال الثاني:** _(ما تفعله بالعيوب التي لا تُحرّك النتيجة؟)_

</div>

## Save your artefacts

Two files, and they are the pair the capstone asks for:

- **`cv_results.parquet`** — one row per fold per configuration: sample size, penalty, `C`, fold
  number, train score, validation score. Per-fold rows and not means, because a mean cannot be
  un-averaged later and the spread is the part you will want.
- **`pipeline.joblib`** — saved in task 2.6. The fitted object: preprocessing and model together.

<div dir="rtl" align="right">

## احفظ مخرجاتك

ملفان، وهما الزوج الذي يطلبه مشروع التخرّج:

- **`cv_results.parquet`** وفيه صف لكل طيّة لكل إعداد: حجم العيّنة ونوع التنظيم وقيمة `C` ورقم الطيّة
  ونتيجة التدريب ونتيجة التحقّق. صفوف الطيّات لا المتوسطات، لأن المتوسط لا يمكن تفكيكه لاحقًا ومدى
  التفاوت هو ما ستريده.
- **`pipeline.joblib`** المحفوظ في المهمة السادسة، وهو الكائن المُدرَّب: المعالجة المسبقة والنموذج معًا.

</div>

In [ ]:
out = ARTEFACT_DIR / "cv_results.parquet"
cv_results.to_parquet(out, index=False)

reloaded_results = pd.read_parquet(out)
print(f"Saved {out}")
print(f"{len(reloaded_results):,} rows "
      f"({reloaded_results['C'].nunique()} C values x "
      f"{reloaded_results['penalty'].nunique()} penalties x "
      f"{reloaded_results['sample'].nunique()} sample sizes x "
      f"{reloaded_results['fold'].nunique()} folds)")
print(f"columns: {list(reloaded_results.columns)}")
print(f"\nand pipeline.joblib is "
      f"{'present' if (ARTEFACT_DIR / 'pipeline.joblib').exists() else 'MISSING'}")

print("\nbest validation score per penalty, on the full sample:")
full = reloaded_results[reloaded_results["sample"] == "full (7,043 rows)"]
best = (full.groupby(["penalty", "C"])["val_score"].mean()
            .reset_index().sort_values("val_score", ascending=False)
            .groupby("penalty").head(1))
print(best.to_string(index=False, float_format=lambda v: f"{v:.4f}"))

## Sanity check

Run this last. Every check that fails tells you what to fix and why.

The fifth one is the strangest assert in this course: it passes when you have **successfully
cheated**. If it fails, your leak did not work, and you should find out why before you believe
anything else in this notebook.

<div dir="rtl" align="right">

## فحص النتائج

شغّل هذه الخلية أخيرًا. كل فحص يفشل يخبرك بما يجب إصلاحه ولماذا.

والفحص الخامس أغرب تحقّق في هذه الدورة: فهو ينجح عندما **تغشّ بنجاح**. فإن فشل فتسريبك لم يعمل، وعليك
أن تعرف السبب قبل أن تصدّق أي شيء آخر في هذا الدفتر.

</div>

In [ ]:
# --- Sanity checks ----------------------------------------------------------------

check("preprocess" in final_pipeline.named_steps and "model" in final_pipeline.named_steps,
      f"the pipeline needs a preprocessing step and an estimator step, it has "
      f"{list(final_pipeline.named_steps)}",
      f"يحتاج خطّ المعالجة خطوة معالجة مسبقة وخطوة مُقدِّر، والموجود "
      f"{list(final_pipeline.named_steps)}")

check(len(cv_scores) == 5,
      f"cross-validation should produce exactly 5 scores, got {len(cv_scores)}",
      f"يجب أن يُنتج التحقّق المتقاطع خمس نتائج بالضبط، والناتج {len(cv_scores)}")

check(cv_scores.std() > 0,
      f"the reported standard deviation must be greater than zero — five identical folds "
      f"means the folds are not different data (std {cv_scores.std():.6f})",
      f"يجب أن يكون الانحراف المعياري المعروض أكبر من صفر — فتطابق الطيّات الخمس يعني أنها "
      f"ليست بيانات مختلفة (الانحراف {cv_scores.std():.6f})")

check(zeros_at_strongest > 0,
      f"at the strongest L1 penalty at least one coefficient must be EXACTLY zero — that is "
      f"the difference between L1 and L2. Got {zeros_at_strongest} zeros.",
      f"عند أشدّ عقوبة L1 يجب أن يكون معامل واحد على الأقل صفرًا **تامًّا**، وهذا هو الفرق "
      f"بين L1 وL2. والناتج {zeros_at_strongest} صفرًا.")

check(bug_b.mean() > honest.mean() + 0.05,
      f"the leaking version must score HIGHER than the honest pipeline "
      f"({bug_b.mean():.4f} vs {honest.mean():.4f}) — passing this check means you "
      f"successfully cheated, which is the point of task 2.5",
      f"يجب أن تكون النسخة المُسرِّبة **أعلى** من خطّ المعالجة الصادق "
      f"({bug_b.mean():.4f} مقابل {honest.mean():.4f}) — والنجاح في هذا الفحص يعني أنك غششت "
      f"بنجاح، وهذا هو المقصود من المهمة الخامسة")

check(np.array_equal(before_saving, after_loading),
      "the reloaded pipeline must predict identically to the original, not approximately — "
      f"max difference {np.abs(before_saving - after_loading).max():.2e}",
      "يجب أن يتنبّأ خطّ المعالجة المُعاد تحميله تنبّؤًا مطابقًا للأصل لا تقريبيًا — "
      f"وأقصى فرق {np.abs(before_saving - after_loading).max():.2e}")

check((ARTEFACT_DIR / "pipeline.joblib").exists()
      and len(reloaded_results) == len(cv_results) > 0,
      f"both artefacts must be on disk: pipeline.joblib "
      f"{(ARTEFACT_DIR / 'pipeline.joblib').exists()}, cv_results.parquet with "
      f"{len(reloaded_results)} rows",
      f"يجب أن يكون الأثران على القرص: pipeline.joblib "
      f"{(ARTEFACT_DIR / 'pipeline.joblib').exists()} وcv_results.parquet بـ "
      f"{len(reloaded_results)} صفًا")

report()

## What's next

That is week 2. You arrived with data that came ready to model and you leave able to take a hostile
file and make it into a defensible one — and, more importantly, able to say **which** of your
decisions was worth anything and by how much.

Next week (**W3D1**) the models change: decision trees and ensembles, which do not care about your
scalers at all and which draw the axis-aligned rectangles that W2D2's scatter plot was asking for.
Everything you built this week survives the change, because `Pipeline` does not care what estimator
sits in its last step.

Two things travel with you:

- **`pipeline.joblib`** is the shape of the capstone's requirement 4. W8D1 loads one and serves it.
- **`0.843 ± 0.011`** is the shape of every performance claim you make from now on. A mean without a
  spread is not a result.

<div dir="rtl" align="right">

## ماذا بعد

هذا هو الأسبوع الثاني. وصلت وبياناتك جاهزة للنمذجة، وتخرج قادرًا على أخذ ملف معطوب وتحويله إلى ملف
قابل للدفاع — والأهم أنك قادر على قول **أيّ** قراراتك كان ذا قيمة وبأي مقدار.

وفي الأسبوع القادم (**الأسبوع ٣ اليوم ١**) تتغيّر النماذج: أشجار القرار والمجموعات، وهي لا تهتمّ
بمُقيِّساتك إطلاقًا وترسم المستطيلات الموازية للمحاور التي كان رسم الانتشار في اليوم الثاني يطلبها. وكل
ما بنيته هذا الأسبوع يصمد أمام التغيير، لأن `Pipeline` لا يهتمّ بأي مُقدِّر يجلس في خطوته الأخيرة.

وشيئان يسافران معك:

- **`pipeline.joblib`** هو شكل المتطلّب الرابع لمشروع التخرّج، ويُحمّله الأسبوع الثامن اليوم الأول ويخدمه.
- **`0.843 ± 0.011`** هو شكل كل ادعاء أداء تقوله من الآن. فالمتوسط بلا مدى تفاوت ليس نتيجة.

</div>